[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_18/2_sql.ipynb)

# Day 18: SQL (medium)

**How to work:** Read the tables first (columns, one row = what?). Write the query in your head in the order FROM, WHERE, GROUP BY, HAVING, SELECT, ORDER BY. Check the result for duplicates and NULLs before you open the solution.

**Today (30 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | Which channel brings valuable users? | medium | 6 |
| Q2 | Cookie Cats: more than one retention number | medium | 6 |
| Q3 | Genre mix of the biggest markets | medium | 6 |
| Q4 | Average order value, two ways | medium | 7 |
| Q5 | Bought within two weeks of signup (review) | medium | 5 |

**Handbook:** [SQL](https://mahsa-agz.github.io/ds-handbook/#sql), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Setup

Run this cell once before the questions.

In [ ]:
import sqlite3
import pandas as pd

def run(q, con=None):
    """Runs a query and shows the result as a table (SELECT / WITH); other statements just run."""
    con = con or db
    if not q.strip():
        print("Write your query between the triple quotes, then run the cell.")
        return None
    if q.strip().lower().startswith(("select", "with")):
        return pd.read_sql_query(q, con)
    con.executescript(q)
    print("done")

# Finds the data folder (local copy of the repository, or /content/data in Colab) and downloads a missing file
# from the ds-handbook repository on GitHub (or from its original source).
import os, sys, urllib.request

RAW = "https://raw.githubusercontent.com/mahsa-agz/mahsa-agz.github.io/main/ds-handbook/exam_prep/data/"

def _find_data():
    if "google.colab" in sys.modules:
        os.makedirs("/content/data", exist_ok=True)
        return "/content/data"
    for p in ("../data", "data", "exam_prep/data"):
        if os.path.isdir(p):
            return p
    os.makedirs("data", exist_ok=True)
    return "data"

DATA = _find_data()
print("data folder:", DATA)

def data_path(name, url=None):
    """Local path of a data file; downloads it from the repository (or from url) if it is not there yet."""
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        for source in (RAW + name, url):
            if not source:
                continue
            try:
                print("downloading", name)
                urllib.request.urlretrieve(source, path)
                break
            except Exception as e:
                print("  could not download from", source, "->", e)
    return path

db = sqlite3.connect(':memory:')

# Real data: the Chinook digital music store (copied into the in-memory db).
_src = sqlite3.connect(data_path("chinook.sqlite", "https://github.com/lerocha/chinook-database/releases/download/v1.4.5/Chinook_Sqlite.sqlite"))
_src.backup(db)
_src.close()

# Made-up shopping-app tables (generated with a fixed seed, so everyone gets the same rows).
import numpy as np

def make_app_tables(con, seed=7):
    rs = np.random.RandomState(seed)
    start, end = pd.Timestamp("2024-01-01"), pd.Timestamp("2024-03-31")
    n = 1500
    countries, c_p = ["US", "KR", "BR", "ID", "DE"], [0.30, 0.25, 0.20, 0.15, 0.10]
    platforms, p_p = ["ios", "android", "web"], [0.40, 0.45, 0.15]
    channels, ch_p = ["organic", "paid_social", "search", "referral"], [0.35, 0.30, 0.20, 0.15]
    signup_off = np.sort(rs.randint(0, 60, n))                     # day offsets 0..59 (Jan 1 to Feb 29)
    country = rs.choice(len(countries), n, p=c_p)
    platform = rs.choice(len(platforms), n, p=p_p)
    channel = rs.choice(len(channels), n, p=ch_p)
    referred_by = [None] * n
    for i in range(n):
        if channels[channel[i]] == "referral":
            earlier = np.nonzero(signup_off[:i] < signup_off[i])[0]
            if len(earlier):
                referred_by[i] = int(earlier[rs.randint(len(earlier))]) + 1
            else:
                channel[i] = 0                                       # nobody to refer them: organic
    users = pd.DataFrame({
        "user_id": np.arange(1, n + 1),
        "signup_date": [(start + pd.Timedelta(days=int(d))).strftime("%Y-%m-%d") for d in signup_off],
        "country": [countries[c] for c in country],
        "platform": [platforms[p] for p in platform],
        "channel": [channels[c] for c in channel],
        "referred_by": pd.array(referred_by, dtype="Int64"),
    })
    plat_boost = {"ios": 0.08, "android": 0.0, "web": -0.10}
    chan_boost = {"organic": 0.05, "paid_social": -0.08, "search": 0.0, "referral": 0.08}
    act, ev, buys = [], [], []
    total_days = (end - start).days + 1
    for i in range(n):
        uid = i + 1
        p = rs.beta(2, 3) + plat_boost[platforms[platform[i]]] + chan_boost[channels[channel[i]]]
        p = min(max(p, 0.03), 0.9)
        life = 1 if rs.rand() < 0.30 else 1 + int(rs.exponential(35))   # days until the user stops coming back
        buyer = rs.rand() < 0.55
        prev = True
        for t in range(0, total_days - signup_off[i]):
            if t == 0:
                active = True
            elif t >= life:
                active = rs.rand() < 0.01                                 # rare comeback after churn
            else:
                active = rs.rand() < (min(p + 0.25, 0.95) if prev else p * 0.7)
            prev = active
            if not active:
                continue
            day = start + pd.Timedelta(days=int(signup_off[i] + t))
            sessions = 1 + rs.poisson(0.8)
            minutes = int(sum(rs.randint(2, 25) for _ in range(sessions)))
            act.append((uid, day.strftime("%Y-%m-%d"), sessions, minutes))
            clock = day + pd.Timedelta(seconds=int(rs.randint(6 * 3600, 22 * 3600)))
            if rs.rand() < 0.65:
                steps = ["view_item"]
                if rs.rand() < 0.40:
                    steps.append("add_to_cart")
                    if rs.rand() < 0.55:
                        steps.append("checkout")
                        if buyer and rs.rand() < 0.75:
                            steps.append("purchase")
                elif rs.rand() < 0.03:
                    steps = ["add_to_cart"]                               # from the wishlist, no view that day
                for s in steps:
                    clock = clock + pd.Timedelta(seconds=int(rs.randint(15, 900)))
                    ev.append((uid, clock.strftime("%Y-%m-%d %H:%M:%S"), s))
                    if s == "purchase":
                        amount = round(float(rs.gamma(2.0, 18.0)) + 5, 2)
                        status = "refunded" if rs.rand() < 0.07 else "completed"
                        buys.append((uid, clock.strftime("%Y-%m-%d %H:%M:%S"), amount, status))
    activity = pd.DataFrame(act, columns=["user_id", "activity_date", "sessions", "minutes"])
    events = pd.DataFrame(ev, columns=["user_id", "event_time", "event_name"])
    events = events.sort_values(["event_time", "user_id"], kind="mergesort").reset_index(drop=True)
    events.insert(0, "event_id", np.arange(1, len(events) + 1))
    purchases = pd.DataFrame(buys, columns=["user_id", "order_time", "amount", "status"])
    purchases = purchases.sort_values(["order_time", "user_id"], kind="mergesort").reset_index(drop=True)
    purchases.insert(0, "order_id", np.arange(1, len(purchases) + 1))
    # raw event log as it arrives from the apps: retries create exact copies, double taps create near copies
    raw = events.copy()
    dup = raw[rs.rand(len(raw)) < 0.04]
    tap = raw[rs.rand(len(raw)) < 0.01].copy()
    tap["event_time"] = (pd.to_datetime(tap["event_time"]) +
                         pd.to_timedelta(rs.randint(1, 4, len(tap)), unit="s")).dt.strftime("%Y-%m-%d %H:%M:%S")
    raw = pd.concat([raw, dup, tap]).sort_values(["event_time", "user_id"], kind="mergesort").reset_index(drop=True)
    raw = raw.drop(columns="event_id")
    raw.insert(0, "raw_id", np.arange(1, len(raw) + 1))
    raw["received_at"] = (pd.to_datetime(raw["event_time"]) +
                          pd.to_timedelta(rs.randint(1, 600, len(raw)), unit="s")).dt.strftime("%Y-%m-%d %H:%M:%S")
    for name, df in [("app_users", users), ("daily_activity", activity), ("app_events", events),
                     ("purchases", purchases), ("app_events_raw", raw)]:
        df.to_sql(name, con, index=False, if_exists="replace")

make_app_tables(db)

# Real data: UCI Online Retail (UK online shop, 2010-12 to 2011-12; CC BY 4.0).
def online_retail():
    path = os.path.join(DATA, "online_retail.csv.gz")
    if not os.path.exists(path):
        import io, zipfile
        print("downloading Online Retail (about 23 MB, then converting the xlsx)")
        z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen("https://archive.ics.uci.edu/static/public/352/online+retail.zip").read()))
        pd.read_excel(io.BytesIO(z.read("Online Retail.xlsx"))).to_csv(path, index=False, compression="gzip")
    return pd.read_csv(path, dtype={"InvoiceNo": str, "StockCode": str})

_o = online_retail()
_o.columns = ["invoice_no", "stock_code", "description", "quantity", "invoice_date", "unit_price", "customer_id",
              "country"]
_o["customer_id"] = _o["customer_id"].astype("Int64")
_o["invoice_date"] = pd.to_datetime(_o["invoice_date"]).dt.strftime("%Y-%m-%d %H:%M:%S")
_o.to_sql("retail", db, index=False)

# Real data: Cookie Cats mobile game A/B test (educational use).
pd.read_csv(data_path("cookie_cats.csv", "https://raw.githubusercontent.com/robguilarr/ab_testing_cookie_cats/main/datasets/cookie_cats.csv")).to_sql("cookie_cats", db, index=False)

print("tables:", ", ".join(r[0] for r in db.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name")))

## Tables

**Real tables: Chinook** (a digital music store, 2009 to 2013; MIT license, lerocha/chinook-database):

| Table | One row is | Key columns |
|---|---|---|
| `Customer` | one customer (59) | `CustomerId`, `FirstName`, `LastName`, `Company` (often NULL), `City`, `State` (often NULL), `Country`, `Email`, `SupportRepId` |
| `Employee` | one employee (8) | `EmployeeId`, `FirstName`, `LastName`, `Title`, `ReportsTo` (manager id, NULL for the boss), `HireDate`, `City` |
| `Invoice` | one purchase (412) | `InvoiceId`, `CustomerId`, `InvoiceDate`, `BillingCity`, `BillingCountry`, `Total` |
| `InvoiceLine` | one track on an invoice (2,240) | `InvoiceLineId`, `InvoiceId`, `TrackId`, `UnitPrice`, `Quantity` |
| `Track` | one song or video (3,503) | `TrackId`, `Name`, `AlbumId`, `MediaTypeId`, `GenreId`, `Composer` (can be NULL), `Milliseconds`, `Bytes`, `UnitPrice` |
| `Album` / `Artist` | one album (347) / artist (275) | `AlbumId`, `Title`, `ArtistId` / `ArtistId`, `Name` |
| `Genre` / `MediaType` | one genre (25) / format (5) | `GenreId`, `Name` / `MediaTypeId`, `Name` |
| `Playlist` / `PlaylistTrack` | one playlist (18) / one track in a playlist (8,715) | `PlaylistId`, `Name` / `PlaylistId`, `TrackId` |

**Made-up shopping-app tables** (invented for practice, generated with a fixed seed: 1,500 users signed up
2024-01-01 to 2024-02-29, activity until 2024-03-31):

| Table | One row is | Columns |
|---|---|---|
| `app_users` | one user | `user_id`, `signup_date` ('YYYY-MM-DD'), `country` (US/KR/BR/ID/DE), `platform` (ios/android/web), `channel` (organic/paid_social/search/referral), `referred_by` (user_id of the referrer, NULL unless channel = referral) |
| `daily_activity` | one user on one day they opened the app | `user_id`, `activity_date` ('YYYY-MM-DD'), `sessions`, `minutes`. No row = not active that day |
| `app_events` | one shopping event | `event_id`, `user_id`, `event_time` ('YYYY-MM-DD HH:MM:SS'), `event_name` (view_item / add_to_cart / checkout / purchase) |
| `purchases` | one order | `order_id`, `user_id`, `order_time`, `amount` (USD), `status` (completed / refunded) |
| `app_events_raw` | one event as received from the apps (has retries and double taps) | `raw_id`, `user_id`, `event_time`, `event_name`, `received_at` |

**Real table: UCI Online Retail** (a UK online gift shop, 2010-12-01 to 2011-12-09; CC BY 4.0):

| Table | One row is | Columns |
|---|---|---|
| `retail` | one product line of an invoice (541,909) | `invoice_no` (starts with 'C' = cancellation), `stock_code`, `description`, `quantity` (negative for returns), `invoice_date` ('YYYY-MM-DD HH:MM:SS'), `unit_price`, `customer_id` (NULL for about 25% of rows), `country` |

**Real table: Cookie Cats** (a mobile puzzle game moved its first gate from level 30 to level 40 for half of new players):

| Table | One row is | Columns |
|---|---|---|
| `cookie_cats` | one new player (90,189) | `userid`, `version` ('gate_30' control / 'gate_40' treatment), `sum_gamerounds` (rounds in the first week), `retention_1`, `retention_7` (1 = came back on day 1 / day 7) |

The queries run in SQLite. They are written in standard SQL that also works in PostgreSQL; differences (dates, integer division) are noted in the solutions.

---
### Q1. Which channel brings valuable users?

*medium, about 6 min*

Shopping app (made up). Per acquisition `channel` return `users`, `buyers` (users with at least one
completed order), `buyer_pct` (1 decimal), `arpu` (completed revenue per user, all users in the denominator) and
`arppu` (completed revenue per paying user), both with 2 decimals. Order by arpu, highest first.

Example: 4 users, one of whom spent 100: arpu = 25.00, arppu = 100.00.

Follow-up: paid_social has the lowest arpu. What extra number do you need before cutting its budget?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: per-user and per-payer averages. Pattern: aggregate revenue per user first, LEFT JOIN to all users, then ratios with the right denominators (COUNT(*) versus COUNT(r.user_id)) and NULLIF.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE rev: completed revenue per user.
2. app_users LEFT JOIN rev; group by channel.
3. arpu = `COALESCE(SUM(revenue), 0) / COUNT(*)`; arppu = `SUM(revenue) / NULLIF(COUNT(r.user_id), 0)`.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH rev AS (
    SELECT user_id, SUM(amount) AS revenue
    FROM purchases
    WHERE status = 'completed'
    GROUP BY user_id
)
SELECT u.channel,
       COUNT(*)                                          AS users,
       COUNT(r.user_id)                                  AS buyers,
       ROUND(100.0 * COUNT(r.user_id) / COUNT(*), 1)     AS buyer_pct,
       ROUND(COALESCE(SUM(r.revenue), 0) / COUNT(*), 2)  AS arpu,
       ROUND(SUM(r.revenue) / NULLIF(COUNT(r.user_id), 0), 2) AS arppu
FROM app_users u
LEFT JOIN rev r ON r.user_id = u.user_id
GROUP BY u.channel
ORDER BY arpu DESC
```

Result:

| channel | users | buyers | buyer_pct | arpu | arppu |
|---|---|---|---|---|---|
| referral | 227 | 60 | 26.4 | 29.08 | 110.03 |
| organic | 545 | 154 | 28.3 | 28.5 | 100.87 |
| search | 303 | 81 | 26.7 | 22.28 | 83.34 |
| paid_social | 425 | 108 | 25.4 | 20.97 | 82.53 |

**Why:** ARPU = buyer rate x ARPPU, so it mixes how many pay with how much they pay. Referral users pay the most per payer; paid_social has both a lower buyer rate and a lower ARPPU. Pre-aggregating revenue per user before the join keeps one row per user, so COUNT(*) is the number of users. NULLIF(x, 0) turns a zero denominator into NULL, so a channel without buyers returns NULL instead of an error. Follow-up: the cost per acquired user (CAC). A low-ARPU channel can still be the most profitable if it is cheap, and the comparison should use the same observation window per user.

**Complexity:** Group by over purchases, a join to 1,500 users, a group by on 4 channels.

**Common mistakes:** Joining raw purchases to users and then COUNT(*) (counts orders, not users). Dividing by buyers for ARPU. Integer division in PostgreSQL when both sides are integers (`buyers / users` is 0).

**Learn more:** [sql-ratios](https://mahsa-agz.github.io/ds-handbook/#sql-ratios), [sql-left-join-nulls](https://mahsa-agz.github.io/ds-handbook/#sql-left-join-nulls), [stats-product-metrics](https://mahsa-agz.github.io/ds-handbook/#stats-product-metrics)

</details>

---
### Q2. Cookie Cats: more than one retention number

*medium, about 6 min*

`cookie_cats` (real mobile-game test; `retention_1` and `retention_7` are 0/1). Per `version` return
`players`, `d1_pct`, `d7_pct`, `d7_given_d1_pct` (of players retained on day 1, the percent also retained on day 7)
and `d7_pct_30plus` (day 7 retention among players with at least 30 rounds), all with 2 decimals.

Example: 10 players, 4 retained on day 1 and 2 of those on day 7: d7_given_d1_pct = 50.00.

Why can `d7_pct_30plus` be misleading as an experiment metric here?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Pattern: rates as averages of 0/1 columns; a conditional rate has a conditional denominator (SUM over the condition, or AVG of a CASE without ELSE).

</details>

<details><summary><b>Hint 2</b></summary>

1. `AVG(retention_1)` is the D1 rate.
2. Conditional: `SUM(retention_7 * retention_1) / NULLIF(SUM(retention_1), 0)`.
3. Subgroup: `AVG(CASE WHEN sum_gamerounds >= 30 THEN retention_7 END)`.

</details>

<details><summary><b>Solution</b></summary>

```sql
SELECT version,
       COUNT(*) AS players,
       ROUND(100.0 * AVG(retention_1), 2) AS d1_pct,
       ROUND(100.0 * AVG(retention_7), 2) AS d7_pct,
       ROUND(100.0 * SUM(retention_7 * retention_1) / NULLIF(SUM(retention_1), 0), 2) AS d7_given_d1_pct,
       ROUND(100.0 * AVG(CASE WHEN sum_gamerounds >= 30 THEN retention_7 END), 2) AS d7_pct_30plus
FROM cookie_cats
GROUP BY version
```

Result:

| version | players | d1_pct | d7_pct | d7_given_d1_pct | d7_pct_30plus |
|---|---|---|---|---|---|
| gate_30 | 44700 | 44.82 | 19.02 | 33.32 | 43.88 |
| gate_40 | 45489 | 44.23 | 18.2 | 32.34 | 43.0 |

**Why:** AVG of a 0/1 column is a proportion; multiplying two 0/1 columns is an AND. Moving the gate to level 40 lowers D7 retention from 19.02% to 18.20%, and D1 slightly. `d7_pct_30plus` conditions on a post-treatment variable: the gate itself changes how many rounds people play, so the players with 30+ rounds are a different mix in each arm and the comparison is no longer randomised. Use it to explore, not to decide. In PostgreSQL booleans need a cast: `AVG(retention_7::int)`.

**Complexity:** One pass over 90,189 rows.

**Common mistakes:** `ELSE 0` in the subgroup average (adds players with fewer rounds as zeros). Dividing D7 by all players when the question asks "given D1". Treating a subgroup filtered on behaviour after assignment as a fair comparison.

**Learn more:** [sql-ratios](https://mahsa-agz.github.io/ds-handbook/#sql-ratios), [sql-case-when](https://mahsa-agz.github.io/ds-handbook/#sql-case-when), [stats-ab-pitfalls](https://mahsa-agz.github.io/ds-handbook/#stats-ab-pitfalls)

</details>

---
### Q3. Genre mix of the biggest markets

*medium, about 6 min*

Chinook. For the **5 customer countries with the most revenue**, return `country`, `revenue` and the
**share of that country's revenue** coming from Rock, Latin, Metal and everything else (`rock_pct`, `latin_pct`,
`metal_pct`, `other_pct`, 1 decimal; the four add up to about 100).

Example: a country with revenue 200 of which Rock is 70 has rock_pct = 35.0.

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Pattern: part / whole inside a group: `SUM(CASE WHEN genre = 'Rock' THEN amount ELSE 0 END) / SUM(amount)` per country. The top 5 countries come from a separate CTE.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE lines: country, genre, amount (UnitPrice * Quantity), with the joins.
2. CTE top5: countries ordered by SUM(amount) DESC LIMIT 5.
3. Group lines by country (only top5) with the four conditional sums divided by the total.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH lines AS (
    SELECT c.Country AS country, g.Name AS genre, il.UnitPrice * il.Quantity AS amount
    FROM InvoiceLine il
    JOIN Invoice i  ON i.InvoiceId = il.InvoiceId
    JOIN Customer c ON c.CustomerId = i.CustomerId
    JOIN Track t    ON t.TrackId = il.TrackId
    JOIN Genre g    ON g.GenreId = t.GenreId
), top5 AS (
    SELECT country FROM lines GROUP BY country ORDER BY SUM(amount) DESC LIMIT 5
)
SELECT country,
       ROUND(SUM(amount), 2) AS revenue,
       ROUND(100.0 * SUM(CASE WHEN genre = 'Rock'  THEN amount ELSE 0 END) / SUM(amount), 1) AS rock_pct,
       ROUND(100.0 * SUM(CASE WHEN genre = 'Latin' THEN amount ELSE 0 END) / SUM(amount), 1) AS latin_pct,
       ROUND(100.0 * SUM(CASE WHEN genre = 'Metal' THEN amount ELSE 0 END) / SUM(amount), 1) AS metal_pct,
       ROUND(100.0 * SUM(CASE WHEN genre NOT IN ('Rock', 'Latin', 'Metal') THEN amount ELSE 0 END)
             / SUM(amount), 1) AS other_pct
FROM lines
WHERE country IN (SELECT country FROM top5)
GROUP BY country
ORDER BY revenue DESC
```

Result:

| country | revenue | rock_pct | latin_pct | metal_pct | other_pct |
|---|---|---|---|---|---|
| USA | 523.06 | 29.7 | 17.2 | 12.1 | 40.9 |
| Canada | 303.96 | 34.8 | 19.5 | 13.0 | 32.6 |
| France | 195.1 | 33.0 | 13.2 | 10.1 | 43.7 |
| Brazil | 190.1 | 42.2 | 27.6 | 7.8 | 22.4 |
| Germany | 156.48 | 39.2 | 11.4 | 15.8 | 33.6 |

**Why:** Conditional sums over the same group give numerator and denominator in one pass, so the shares are consistent and add to 100. Brazil leans most on Rock (42.2%) and Latin (27.6%); the USA is the most diverse (40.9% other). An alternative is one row per (country, genre) with `amount / SUM(amount) OVER (PARTITION BY country)`, better when there are many categories.

**Complexity:** One pass over 2,240 lines plus the top-5 subquery.

**Common mistakes:** Dividing by the global revenue instead of the country revenue. Using AVG(CASE ...) (averages line amounts, not a share). Forgetting the 'other' bucket so the columns do not add up.

**Learn more:** [sql-ratios](https://mahsa-agz.github.io/ds-handbook/#sql-ratios), [sql-case-when](https://mahsa-agz.github.io/ds-handbook/#sql-case-when), [sql-pivot](https://mahsa-agz.github.io/ds-handbook/#sql-pivot)

</details>

---
### Q4. Average order value, two ways

*medium, about 7 min*

Online Retail (real). Use non-cancelled invoices with a `customer_id`. An order is one `invoice_no`;
its value is `SUM(quantity * unit_price)`. For countries with **at least 25 customers**, return `customers`, `orders`,
`aov_all_orders` (total revenue / total orders) and `avg_customer_aov` (compute each customer's own AOV, then average
those), 2 decimals, most customers first.

Example: customer A has 1 order of 1,000; customer B has 9 orders of 100. aov_all_orders = 1,900 / 10 = 190;
avg_customer_aov = (1,000 + 100) / 2 = 550.

Which one would you put on a dashboard, and why do France and Germany swap places between the two columns?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: "average of a ratio". Pattern: ratio of sums (weights each order equally) versus average of per-unit ratios (weights each customer equally). Compute both from a per-customer table.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE inv: one row per invoice with its value.
2. CTE per_cust: revenue and order count per (country, customer).
3. `SUM(revenue) / SUM(orders)` and `AVG(revenue / orders)` per country, HAVING COUNT(*) >= 25.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH inv AS (
    SELECT invoice_no, customer_id, country, SUM(quantity * unit_price) AS amount
    FROM retail
    WHERE customer_id IS NOT NULL AND invoice_no NOT LIKE 'C%'
    GROUP BY invoice_no, customer_id, country
), per_cust AS (
    SELECT country, customer_id, SUM(amount) AS revenue, COUNT(*) AS orders
    FROM inv
    GROUP BY country, customer_id
)
SELECT country,
       COUNT(*)                            AS customers,
       SUM(orders)                         AS orders,
       ROUND(SUM(revenue) / SUM(orders), 2) AS aov_all_orders,
       ROUND(AVG(revenue / orders), 2)      AS avg_customer_aov
FROM per_cust
GROUP BY country
HAVING COUNT(*) >= 25
ORDER BY customers DESC
```

Result:

| country | customers | orders | aov_all_orders | avg_customer_aov |
|---|---|---|---|---|
| United Kingdom | 3921 | 16649 | 438.97 | 393.22 |
| Germany | 94 | 457 | 500.8 | 542.89 |
| France | 87 | 389 | 537.34 | 464.41 |
| Spain | 30 | 90 | 684.19 | 635.09 |
| Belgium | 25 | 98 | 420.37 | 462.11 |

**Why:** The two answer different questions. Ratio of sums = "what is a typical order worth" (customers with many orders weigh more). Average of ratios = "what is a typical customer's order worth" (each customer weighs 1). France has a higher order-weighted AOV (537.34 versus 500.80), Germany a higher customer-weighted one (542.89 versus 464.41): in France the frequent buyers place big orders, in Germany some rare buyers do. For a revenue dashboard use the ratio of sums (it adds up to total revenue); always say which one you report. In A/B tests a ratio metric needs the delta method because the unit of randomisation (user) differs from the unit of the metric (order).

**Complexity:** Two group bys over about 400k lines.

**Common mistakes:** `AVG(amount)` on line level (that is average line value, not order value). Mixing both definitions across reports. `revenue / orders` with integer columns in PostgreSQL (truncates).

**Learn more:** [sql-ratios](https://mahsa-agz.github.io/ds-handbook/#sql-ratios), [stats-ratio-metrics](https://mahsa-agz.github.io/ds-handbook/#stats-ratio-metrics)

</details>

---
### Q5. Bought within two weeks of signup (review)

*medium, about 5 min*

Shopping app. Per signup month (`cohort`), return `users`, `bought_14d` (first order of any status
placed less than 14 days after the signup date, i.e. before `date(signup_date, '+14 day')`) and `pct_14d`
(1 decimal), plus `pct_ever` (bought at any time in the data).

Why is `pct_ever` not a fair comparison between the January and February cohorts, while `pct_14d` is?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Pattern: a time-bounded funnel: first purchase time per user, LEFT JOIN to users, a CASE on the time window.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE first_buy: MIN(order_time) per user.
2. app_users LEFT JOIN first_buy; `SUM(CASE WHEN f.t < date(u.signup_date, '+14 day') THEN 1 ELSE 0 END)`.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH first_buy AS (
    SELECT user_id, MIN(order_time) AS t FROM purchases GROUP BY user_id
)
SELECT substr(u.signup_date, 1, 7) AS cohort,
       COUNT(*) AS users,
       SUM(CASE WHEN f.t < date(u.signup_date, '+14 day') THEN 1 ELSE 0 END) AS bought_14d,
       ROUND(100.0 * SUM(CASE WHEN f.t < date(u.signup_date, '+14 day') THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_14d,
       ROUND(100.0 * SUM(CASE WHEN f.t IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_ever
FROM app_users u
LEFT JOIN first_buy f ON f.user_id = u.user_id
GROUP BY substr(u.signup_date, 1, 7)
ORDER BY cohort
```

Result:

| cohort | users | bought_14d | pct_14d | pct_ever |
|---|---|---|---|---|
| 2024-01 | 779 | 160 | 20.5 | 29.3 |
| 2024-02 | 721 | 142 | 19.7 | 26.9 |

**Why:** A NULL first purchase makes the CASE false, so non-buyers count as 0. January users had about a month longer to buy, so `pct_ever` (29.3% versus 26.9%) is biased toward older cohorts. A fixed window gives every user the same chance: 20.5% versus 19.7%, nearly equal. Comparing a timestamp with a date string works because 'YYYY-MM-DD' sorts before 'YYYY-MM-DD HH:MM:SS' of the same day.

**Complexity:** Group by over purchases, a join to users.

**Common mistakes:** Writing `<= date(...)` and expecting the whole last day to count: as text '2024-01-15 08:00:00' is greater than '2024-01-15', so it is excluded. Comparing cohorts with open-ended windows. Inner join (drops non-buyers from the denominator).

**Learn more:** [sql-funnels](https://mahsa-agz.github.io/ds-handbook/#sql-funnels), [sql-retention](https://mahsa-agz.github.io/ds-handbook/#sql-retention), [sql-dates](https://mahsa-agz.github.io/ds-handbook/#sql-dates)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Next:** [Statistics, A/B testing and product](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_18/3_stats.ipynb)